In [12]:
from dotenv import load_dotenv
load_dotenv()
import pandas as pd

# QA
inputs = [
    "For customer-facing applications, which company's models dominate the top rankings?",
    "What percentage of respondents are using RAG in some form?",
    "How often are most respondents updating their models?",
]

outputs = [
    "OpenAI models dominate, with 3 of the top 5 and half of the top 10 most popular models for customer-facing apps.",
    "70% of respondents are using RAG in some form.",
    "More than 50% update their models at least monthly, with 17% doing so weekly.",
]

# Dataset
qa_pairs = [{"question": q, "answer": a} for q, a in zip(inputs, outputs)]
df = pd.DataFrame(qa_pairs)

# Write to csv
csv_path = "C:\\Users\\fredd\\OneDrive\\Documents\\LLMOPs\\data\\goldens.csv"
df.to_csv(csv_path, index=False)


In [ ]:
from langsmith import Client

client = Client()
dataset_name = "AgenticAIReportGoldens"

# Store
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Input and expected output pairs for AgenticAIReport",
)
client.create_examples(
    inputs=[{"question": q} for q in inputs],
    outputs=[{"answer": a} for a in outputs],
    dataset_id=dataset.id,
)

{'example_ids': ['3165a59c-22e8-4c72-8e37-07f69d196e2f',
  '707c46bd-fad6-44f6-bec3-f737caa7a9b2',
  '306bb6e5-72c7-49fd-84d8-4df702343f2e'],
 'count': 3}

In [2]:
import sys
sys.path.append("C:\\Users\\fredd\\OneDrive\\Documents\\LLMOPs")

from pathlib import Path
from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG
import os

# Simple file adapter for local file paths
class LocalFileAdapter:
    """Adapter for local file paths to work with ChatIngestor."""
    def __init__(self, file_path: str):
        self.path = Path(file_path)
        self.name = self.path.name
    
    def getbuffer(self) -> bytes:
        return self.path.read_bytes()


def answer_ai_report_question(
    inputs: dict,
    data_path: str = "C:\\Users\\fredd\\OneDrive\\Documents\\LLMOPs\\data\\notes.txt",
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    k: int = 5
) -> dict:
    """
    Answer questions about the AI Engineering Report using RAG.
    
    Args:
        inputs: Dictionary containing the question, e.g., {"question": "What is RAG?"}
        data_path: Path to the AI Engineering Report text file
        chunk_size: Size of text chunks for splitting
        chunk_overlap: Overlap between chunks
        k: Number of documents to retrieve
    
    Returns:
        Dictionary with the answer, e.g., {"answer": "RAG stands for..."}
    """
    try:
        # Extract question from inputs
        question = inputs.get("question", "")
        if not question:
            return {"answer": "No question provided"}
        
        # Check if file exists
        if not Path(data_path).exists():
            return {"answer": f"Data file not found: {data_path}"}
        
        # Create file adapter
        file_adapter = LocalFileAdapter(data_path)
        
        # Build index using ChatIngestor
        ingestor = ChatIngestor(
            temp_base="data",
            faiss_base="faiss_index",
            use_session_dirs=True
        )
        
        # Build retriever
        ingestor.built_retriver(
            uploaded_files=[file_adapter],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            k=k
        )
        
        # Get session ID and index path
        session_id = ingestor.session_id
        index_path = f"faiss_index/{session_id}"
        
        # Create RAG instance and load retriever
        rag = ConversationalRAG(session_id=session_id)
        rag.load_retriever_from_faiss(
            index_path=index_path,
            k=k,
            index_name=os.getenv("FAISS_INDEX_NAME", "index")
        )
        
        # Get answer
        answer = rag.invoke(question, chat_history=[])
        
        return {"answer": answer}
        
    except Exception as e:
        return {"answer": f"Error: {str(e)}"}


In [15]:
!uv pip install -U "langsmith>=0.3.13" langchain langchain-core langchain-community langchain-groq


Using Python 3.11.14 environment at: C:\Users\fredd\.conda\envs\llmops311
Resolved 57 packages in 1.83s
 Downloaded langchain-community
Prepared 13 packages in 7.38s
Uninstalled 6 packages in 1.56s
Installed 13 packages in 751ms
 - langchain==0.3.27
 + langchain==1.2.2
 + langchain-classic==1.0.1
 - langchain-community==0.3.27
 + langchain-community==0.4.1
 - langchain-core==0.3.72
 + langchain-core==1.2.6
 - langchain-groq==0.3.6
 + langchain-groq==1.1.1
 - langchain-text-splitters==0.3.9
 + langchain-text-splitters==1.1.0
 + langgraph==1.0.5
 + langgraph-checkpoint==3.0.1
 + langgraph-prebuilt==1.0.5
 + langgraph-sdk==0.3.1
 + ormsgpack==1.12.1
 - python-dotenv==1.1.1
 + python-dotenv==1.2.1
 + xxhash==3.6.0


In [4]:
# Test the function with a sample question
test_input = {"question": "For customer-facing applications, which company's models dominate the top rankings?"}
result = answer_ai_report_question(test_input)
print("Question:", test_input["question"])
print("\nAnswer:", result["answer"])
        

{"timestamp": "2026-01-07T23:21:58.704632Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-01-07T23:21:58.705635Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-01-07T23:21:58.706632Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_To...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2026-01-07T23:21:58.706632Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-01-07T23:21:58.710655Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260108_002158_a2652aa6", "temp_dir": "data\\session_20260108_002158_a2652aa6", "faiss_dir": "faiss_index\\session_20260108_002158_a2652aa6", "sessionized": true, "timestamp": "2026-01-07T23:21:58.714182Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "notes.txt", "saved_as": "data\\session_20260

Question: For customer-facing applications, which company's models dominate the top rankings?

Answer: I don't know. The provided context does not mention customer-facing applications or rankings.


In [6]:
from langsmith.evaluation import evaluate
# Example: Test with all golden questions
print("Testing all questions from the dataset:\n")
for i, q in enumerate(inputs, 1):
    test_input = {"question": q}
    result = answer_ai_report_question(test_input)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}\n")
    print("-" * 80 + "\n")

{"timestamp": "2026-01-07T23:24:30.904077Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-01-07T23:24:30.907478Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-01-07T23:24:30.908756Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_To...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2026-01-07T23:24:30.910757Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-01-07T23:24:30.916836Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260108_002430_11895a0c", "temp_dir": "data\\session_20260108_002430_11895a0c", "faiss_dir": "faiss_index\\session_20260108_002430_11895a0c", "sessionized": true, "timestamp": "2026-01-07T23:24:30.920844Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "notes.txt", "saved_as": "data\\session_20260

Testing all questions from the dataset:



{"added": 1, "index": "faiss_index\\session_20260108_002430_11895a0c", "timestamp": "2026-01-07T23:24:31.564988Z", "level": "info", "event": "FAISS index updated"}
{"k": 5, "fetch_k": 20, "lambda_mult": 0.5, "timestamp": "2026-01-07T23:24:31.567129Z", "level": "info", "event": "Using MMR search"}
{"timestamp": "2026-01-07T23:24:31.572750Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-01-07T23:24:31.574754Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-01-07T23:24:31.575757Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_To...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2026-01-07T23:24:31.576752Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-01-07T23:24:31.579903Z", "level": "info", "event": "YAML config loaded"}
{"provider": "groq", "model": "llama-3.1-8b-in

Q1: For customer-facing applications, which company's models dominate the top rankings?
A1: I don't know. The provided context does not mention customer-facing applications or rankings.

--------------------------------------------------------------------------------



{"added": 1, "index": "faiss_index\\session_20260108_002433_45f9d720", "timestamp": "2026-01-07T23:24:33.939067Z", "level": "info", "event": "FAISS index updated"}
{"k": 5, "fetch_k": 20, "lambda_mult": 0.5, "timestamp": "2026-01-07T23:24:33.941067Z", "level": "info", "event": "Using MMR search"}
{"timestamp": "2026-01-07T23:24:33.946360Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-01-07T23:24:33.948369Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-01-07T23:24:33.949372Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_To...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2026-01-07T23:24:33.950368Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-01-07T23:24:33.953369Z", "level": "info", "event": "YAML config loaded"}
{"provider": "groq", "model": "llama-3.1-8b-in

Q2: What percentage of respondents are using RAG in some form?
A2: I don't know.

--------------------------------------------------------------------------------



{"added": 1, "index": "faiss_index\\session_20260108_002435_3d659bb1", "timestamp": "2026-01-07T23:24:36.282539Z", "level": "info", "event": "FAISS index updated"}
{"k": 5, "fetch_k": 20, "lambda_mult": 0.5, "timestamp": "2026-01-07T23:24:36.285751Z", "level": "info", "event": "Using MMR search"}
{"timestamp": "2026-01-07T23:24:36.292301Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-01-07T23:24:36.294304Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-01-07T23:24:36.295936Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_To...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2026-01-07T23:24:36.296938Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-01-07T23:24:36.303461Z", "level": "info", "event": "YAML config loaded"}
{"provider": "groq", "model": "llama-3.1-8b-in

Q3: How often are most respondents updating their models?
A3: I don't know.

--------------------------------------------------------------------------------



In [11]:
from langsmith.evaluation import evaluate

# Custom evaluator for QA
def qa_evaluator(run, example):
    predicted = run.outputs.get("answer", "").strip().lower()
    reference = example.outputs.get("answer", "").strip().lower()
    score = 1.0 if predicted == reference else 0.0
    return {"score": score}

# Evaluators
evaluators = [qa_evaluator]
dataset_name = "AgenticAIReportGoldens"

# Run evaluation using our RAG function
experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=evaluators,
    experiment_prefix="testing-agenticAIReport-qa-rag",
    # Experiment metadata
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

View the evaluation results for experiment: 'testing-agenticAIReport-qa-rag-0b0fa32b' at:
https://smith.langchain.com/o/53aa6101-de92-4bab-ad24-9bc22621b1c6/datasets/bc32ffca-ab82-4ecf-8e18-d4a8fc675b28/compare?selectedSessions=2e3bf205-e5fb-4434-942f-f5fbff1c5dbf




0it [00:00, ?it/s]{"timestamp": "2026-01-07T23:29:52.256629Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-01-07T23:29:52.258636Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-01-07T23:29:52.258636Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_To...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2026-01-07T23:29:52.259637Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-01-07T23:29:52.264707Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260108_002952_047cfaee", "temp_dir": "data\\session_20260108_002952_047cfaee", "faiss_dir": "faiss_index\\session_20260108_002952_047cfaee", "sessionized": true, "timestamp": "2026-01-07T23:29:52.267717Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "notes.txt", "saved_as": "d

In [18]:
from langsmith import Client
from langchain.evaluation import load_evaluator
from langchain_groq import ChatGroq

# Groq model used as the "LLM-as-judge"
eval_llm = ChatGroq(
    model="llama-3.1-70b-versatile",
    temperature=0,
)

# Start with the simplest evaluator
evaluators = [
    load_evaluator("qa", llm=eval_llm),
]

client = Client()

results = client.evaluate(
    answer_ai_report_question,          # must accept inputs: dict and return outputs (dict or str)
    data="AgenticAIReportGoldens",      # dataset name or UUID :contentReference[oaicite:5]{index=5}
    evaluators=evaluators,
    experiment_prefix="rag-faiss-groq-qa",
    metadata={
        "variant": "RAG with FAISS",
        "judge": "groq/llama-3.1-70b-versatile",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
    max_concurrency=1,
)

print(results)

View the evaluation results for experiment: 'rag-faiss-groq-qa-23ee9d97' at:
https://smith.langchain.com/o/53aa6101-de92-4bab-ad24-9bc22621b1c6/datasets/bc32ffca-ab82-4ecf-8e18-d4a8fc675b28/compare?selectedSessions=f57b2fe3-780e-40af-8b6f-5b8fa0a043d0




0it [00:00, ?it/s]{"timestamp": "2026-01-07T23:39:08.943459Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-01-07T23:39:08.946635Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-01-07T23:39:08.947635Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_To...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2026-01-07T23:39:08.947635Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-01-07T23:39:08.953707Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260108_003908_a8baa558", "temp_dir": "data\\session_20260108_003908_a8baa558", "faiss_dir": "faiss_index\\session_20260108_003908_a8baa558", "sessionized": true, "timestamp": "2026-01-07T23:39:08.957709Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "notes.txt", "saved_as": "d

<ExperimentResults rag-faiss-groq-qa-23ee9d97>


## Custom Evaluator

In [19]:
from langsmith.schemas import Run, Example
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

def correctness_evaluator(run: Run, example: Example) -> dict:
    """
    Custom LLM-as-a-Judge evaluator for correctness.
    
    Correctness means how well the actual model output matches the reference output 
    in terms of factual accuracy, coverage, and meaning.
    
    Args:
        run: The Run object containing the actual outputs
        example: The Example object containing the expected outputs
    
    Returns:
        dict with 'score' (1 for correct, 0 for incorrect) and 'reasoning'
    """
    # Extract actual and expected outputs
    actual_output = run.outputs.get("answer", "")
    expected_output = example.outputs.get("answer", "")
    input_question = example.inputs.get("question", "")
    
    # Define the evaluation prompt
    eval_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an evaluator whose job is to judge correctness.

Correctness means how well the actual model output matches the reference output in terms of factual accuracy, coverage, and meaning.

- If the actual output matches the reference output semantically (even if wording differs), it should be marked correct.
- If the output misses key facts, introduces contradictions, or is factually incorrect, it should be marked incorrect.

Do not penalize for stylistic or formatting differences unless they change meaning."""),
        ("human", """<example>
<input>
{input}
</input>

<output>
Expected Output: {expected_output}

Actual Output: {actual_output}
</output>
</example>

Please grade the following agent run given the input, expected output, and actual output.
Focus only on correctness (semantic and factual alignment).

Respond with:
1. A brief reasoning (1-2 sentences)
2. A final verdict: either "CORRECT" or "INCORRECT"

Format your response as:
Reasoning: [your reasoning]
Verdict: [CORRECT or INCORRECT]""")
    ])
    
    # Initialize LLM (using Gemini as shown in your config)
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-pro",
        temperature=0
    )
    
    # Create chain and invoke
    chain = eval_prompt | llm
    
    try:
        response = chain.invoke({
            "input": input_question,
            "expected_output": expected_output,
            "actual_output": actual_output
        })
        
        response_text = response.content
        
        # Parse the response
        reasoning = ""
        verdict = ""
        
        for line in response_text.split('\n'):
            if line.startswith("Reasoning:"):
                reasoning = line.replace("Reasoning:", "").strip()
            elif line.startswith("Verdict:"):
                verdict = line.replace("Verdict:", "").strip()
        
        # Convert verdict to score (1 for correct, 0 for incorrect)
        score = 1 if "CORRECT" in verdict.upper() else 0
        
        return {
            "key": "correctness",
            "score": score,
            "reasoning": reasoning,
            "comment": f"Verdict: {verdict}"
        }
        
    except Exception as e:
        return {
            "key": "correctness",
            "score": 0,
            "reasoning": f"Error during evaluation: {str(e)}"
        }


In [24]:
from langchain_groq import ChatGroq
from langchain.evaluation import load_evaluator

judge_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
qa_eval = load_evaluator("qa", llm=judge_llm)

def correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    query = inputs.get("question") or inputs.get("query") or ""
    prediction = outputs.get("answer") or outputs.get("result") or ""
    reference = reference_outputs.get("answer") or reference_outputs.get("result") or ""

    try:
        res = qa_eval.evaluate_strings(
            input=query,
            prediction=prediction,
            reference=reference,
        )
        # res often contains keys like "score" and "reasoning" depending on evaluator
        score = res.get("score", None)
        comment = res.get("reasoning", "") or res.get("value", "") or ""
        return {"key": "correctness", "score": score, "comment": comment}
    except Exception as e:
        # still return a valid evaluation object, but mark as failed
        return {"key": "correctness", "score": None, "comment": f"Evaluator error: {e}"}
from langsmith import Client

client = Client()

results = client.evaluate(
    answer_ai_report_question,
    data="AgenticAIReportGoldens",
    evaluators=[correctness_evaluator],
    experiment_prefix="rag-faiss-groq-correctness",
    max_concurrency=1,  # keep low while debugging rate limits
)


View the evaluation results for experiment: 'rag-faiss-groq-correctness-f61ff2f0' at:
https://smith.langchain.com/o/53aa6101-de92-4bab-ad24-9bc22621b1c6/datasets/bc32ffca-ab82-4ecf-8e18-d4a8fc675b28/compare?selectedSessions=f0a5164e-03b1-42be-8c94-11596b555a91




0it [00:00, ?it/s]{"timestamp": "2026-01-07T23:51:50.052565Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-01-07T23:51:50.056093Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-01-07T23:51:50.057093Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_To...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2026-01-07T23:51:50.057093Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-01-07T23:51:50.063611Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260108_005150_b4f4b3f0", "temp_dir": "data\\session_20260108_005150_b4f4b3f0", "faiss_dir": "faiss_index\\session_20260108_005150_b4f4b3f0", "sessionized": true, "timestamp": "2026-01-07T23:51:50.068857Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "notes.txt", "saved_as": "d